Student Name: U G P G Mihiranga
Student ID: IT23153486
Model:Deep Embedded Forest (DEF) model

In [1]:
%pip install pandas scikit-learn numpy tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from sklearn.ensemble import RandomForestClassifier

In [3]:
# --- 1. Load and Clean the Data ---
df = pd.read_csv("data.csv")

# Drop unnecessary columns: 'id' (identifier) and 'Unnamed: 32' (all NaN)
df = df.drop(['id', 'Unnamed: 32'], axis=1)

In [4]:
# --- 2. Target Encoding ---
# Encode 'diagnosis' (Malignant/Benign) to 1/0
label_encoder = LabelEncoder()
df['diagnosis'] = label_encoder.fit_transform(df['diagnosis'])
# M is typically 1, B is typically 0, confirmed in the execution output.

In [5]:
# --- 3. Feature/Target Separation (MISSING STEP ADDED) ---
# Define X (features) as all columns except 'diagnosis'
X = df.drop('diagnosis', axis=1)
# Define y (target) as the 'diagnosis' column
y = df['diagnosis']

In [6]:
# --- 4. Split Data ---
# Stratify ensures the train/test split has the same proportion of M/B
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
# --- 5. Feature Scaling (Crucial for the Deep Learning Component) ---
# Use StandardScaler (Z-score normalization)
scaler = StandardScaler()

# Fit scaler on X_train and transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
# Convert back to NumPy arrays (needed for TensorFlow/Keras)
X_train_scaled = np.array(X_train_scaled)
X_test_scaled = np.array(X_test_scaled)
y_train = np.array(y_train)
y_test = np.array(y_test)

In [9]:
# --- Stage 1: Deep Embedding Network ---

# Define the input shape
input_shape = X_train_scaled.shape[1] # 30 features
embedding_dim = 16 # The size of the compressed feature representation

In [10]:
# 1. Feature Extractor (Embedding) Layers
input_layer = Input(shape=(input_shape,), name='input_features')
h = Dense(64, activation='relu', name='dense_64')(input_layer)
h = Dropout(0.2)(h)
# This bottleneck layer will be the output of our embedder
embedding_layer = Dense(embedding_dim, activation='relu', name='embedding_16')(h)
h = Dropout(0.2)(embedding_layer)

In [11]:
# 2. Classification Head
output_layer = Dense(1, activation='sigmoid', name='output_diagnosis')(h)

# Full Classifier Model
classifier_model = Model(inputs=input_layer, outputs=output_layer)

# Compile and train the classifier model
classifier_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [12]:
# Train the model (e.g., 50-100 epochs with early stopping)
classifier_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=50,
    batch_size=32,
    verbose=1
)


Epoch 1/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 4s 82ms/step - accuracy: 0.6659 - loss: 0.6035 - val_accuracy: 0.9298 - val_loss: 0.4260
Epoch 2/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8835 - loss: 0.3681 - val_accuracy: 0.9386 - val_loss: 0.2700
Epoch 3/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9187 - loss: 0.2587 - val_accuracy: 0.9474 - val_loss: 0.1990
Epoch 4/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9407 - loss: 0.2106 - val_accuracy: 0.9561 - val_loss: 0.1600
Epoch 5/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9473 - loss: 0.1756 - val_accuracy: 0.9561 - val_loss: 0.1353
Epoch 6/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9604 - loss: 0.1546 - val_accuracy: 0.9649 - val_loss: 0.1187
Epoch 7/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9604 - loss: 0.1331 - val_accuracy: 0.9649 - val_loss: 0.1078
Epoch 8/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9670 - loss: 0.1200 - val_accuracy: 0.9649 - val_l

In [13]:
# --- Extract the Feature Embedder ---
# Create a new model that outputs the feature embedding instead of the diagnosis
feature_embedder = Model(inputs=input_layer, outputs=embedding_layer)

# Generate new, embedded features
X_train_embedded = feature_embedder.predict(X_train_scaled)
X_test_embedded = feature_embedder.predict(X_test_scaled)

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
